# Week 4 Exercise - Ruby on Rails code RSpec generator

This notebook implements an RSpec generator for the provided codebase in zip format
using Gradio as UI. This is helpful if you have some code that doesn't have test coverage.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import zipfile
import tempfile
from pathlib import Path

# environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    print("OpenAPI API Key is missing.")
client = OpenAI()


In [ ]:
def procesar_archivos(input_zip):
    # Create a temporary directory to save the files
    with tempfile.TemporaryDirectory() as tmpdir:
        output_path = os.path.join(tmpdir, "resultado.zip")
        extracted_path = os.path.join(tmpdir, "extraidos")
        processed_path = os.path.join(tmpdir, "procesados")
        os.makedirs(processed_path, exist_ok=True)

        # 1. Uncompress the uploaded zip file
        with zipfile.ZipFile(input_zip.name, 'r') as zip_ref:
            zip_ref.extractall(extracted_path)

        # 2. Process each rb file with OpenAI
        for root, _, files in os.walk(extracted_path):
            for file in files:
                if file.endswith(".rb"):
                    input_file_path = os.path.join(root, file)
                    with open(input_file_path, 'r', encoding='utf-8') as f:
                        contenido = f.read()

                    # Llamada a OpenAI (ejemplo: explicar)
                    response = client.chat.completions.create(
                        model="gpt-4.1-mini",
                        messages=[{"role": "user", "content": f"Generate RSpec tests for this code: {contenido}"}]
                    )
                    
                    # Save the result
                    rspec_code = response.choices[0].message.content
                    filename = os.path.splitext(file)[0] + "_rspec.rb"
                    with open(os.path.join(processed_path, f"{filename}"), "w") as f:
                        f.write(rspec_code)

        # 3. Create the output zip file
        with zipfile.ZipFile(output_path, 'w') as zip_out:
            for root, _, files in os.walk(processed_path):
                for file in files:
                    zip_out.write(os.path.join(root, file), file)

        # Important: Gradio needs a persistent path to download the file
        final_zip = os.path.join(os.getcwd(), "generated_rspec.zip")
        import shutil
        shutil.copy(output_path, final_zip)
        
        return final_zip


In [ ]:
# Interfaz de Gradio
demo = gr.Interface(
    fn=procesar_archivos,
    inputs=gr.File(label="Upload your Ruby on Rails codebase", file_types=[".zip"]),
    outputs=gr.File(label="Download your RSpec tests"),
    title="RSPEC generator with OpenAI",
    description="Upload your Ruby on Rails codebase to generate RSpec tests."
)

if __name__ == "__main__":
    demo.launch()
